# GIN Replication — Google Colab (T4 GPU)

Run these cells **in order**, top to bottom.

**First:** `Runtime` → `Change runtime type` → select **T4 GPU** → Save.

**What this notebook does:**
1. Clones the official code for the paper *"How Powerful are Graph Neural Networks?"* (GIN, Xu et al., ICLR 2019)
2. Runs the paper's exact training/evaluation protocol on **IMDB-BINARY** and **IMDB-MULTI** — the paper's own original data (ships inside the repo, no download needed)
3. After you upload two files, runs the same protocol on **our new scraped data** — `IMDBBINARY-NEW` and `IMDBMULTI-NEW`
4. Prints one final summary block — **copy that block back into the chat when it's done**

Each dataset run is 10-fold cross-validation × 350 epochs (the paper's exact protocol). Should be noticeably faster than CPU thanks to the T4. Results are saved after every fold, so if Colab disconnects partway, nothing already-finished is lost.


## 0. Check the GPU is actually connected

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Go to: Runtime > Change runtime type > select GPU (T4) > Save, "
          "then Runtime > Restart session, then re-run this cell.")


## 1. Clone the official GIN repo and install dependencies

In [ ]:
!git clone -q https://github.com/weihua916/powerful-gnns.git
%cd powerful-gnns
!unzip -q -o dataset.zip
!pip install -q networkx tqdm
print("Setup done. Datasets available:", end=" ")
!ls dataset


## 2. Upload our scraped datasets

Run the next cell, then in the file picker select **both** of these files from your computer
(from `new_data/processed/.../` in the project folder):

- `IMDBBINARY-NEW.txt`
- `IMDBMULTI-NEW.txt`

(You do **not** need the `_metadata.json` files for this — just the two `.txt` files.)


In [ ]:
from google.colab import files
import os, shutil

uploaded = files.upload()

for fname in uploaded.keys():
    if not fname.endswith(".txt"):
        print(f"skipping {fname} (not a .txt dataset file)")
        continue
    dsname = fname[:-4]
    os.makedirs(f"dataset/{dsname}", exist_ok=True)
    shutil.move(fname, f"dataset/{dsname}/{fname}")
    print(f"placed dataset/{dsname}/{fname}")


## 3. Training / cross-validation driver

Same protocol the paper describes: run all 10 folds, average the 10 test-accuracy curves epoch-by-epoch, pick the single best epoch, report mean ± std across folds at that epoch.

In [ ]:
%%writefile run_gin.py
import argparse
import json
import re
import subprocess
import sys
import time
from pathlib import Path

RESULTS_DIR = Path("results")
EPOCH_RE = re.compile(r"loss training: ([\d.]+)\naccuracy train: ([\d.]+) test: ([\d.]+)")


def run_fold(dataset, fold_idx, extra_args, epochs):
    cmd = [sys.executable, "main.py", "--dataset", dataset,
           "--fold_idx", str(fold_idx), "--epochs", str(epochs)] + extra_args
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"fold {fold_idx} failed:\n{proc.stderr[-3000:]}")
    matches = EPOCH_RE.findall(proc.stdout)
    if len(matches) != epochs:
        raise RuntimeError(
            f"fold {fold_idx}: expected {epochs} epochs, parsed {len(matches)}\n{proc.stdout[-1000:]}"
        )
    return [{"loss": float(l), "train_acc": float(tr), "test_acc": float(te)} for l, tr, te in matches]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dataset", required=True)
    ap.add_argument("--epochs", type=int, default=350)
    ap.add_argument("--extra", nargs=argparse.REMAINDER, default=[])
    args = ap.parse_args()

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    out_path = RESULTS_DIR / f"{args.dataset}_cv_curves.json"

    all_folds = []
    t0 = time.time()
    for fold_idx in range(10):
        fstart = time.time()
        curve = run_fold(args.dataset, fold_idx, args.extra, args.epochs)
        all_folds.append(curve)
        print(f"[{args.dataset}] fold {fold_idx} done in {time.time() - fstart:.1f}s "
              f"(test_acc={curve[-1]['test_acc']:.4f})", flush=True)
        out_path.write_text(json.dumps(
            {"dataset": args.dataset, "epochs": args.epochs, "extra_args": args.extra, "folds": all_folds},
            indent=2))

    n_epochs = args.epochs
    avg_curve = [sum(fold[e]["test_acc"] for fold in all_folds) / len(all_folds) for e in range(n_epochs)]
    best_epoch = max(range(n_epochs), key=lambda e: avg_curve[e])
    accs = [fold[best_epoch]["test_acc"] for fold in all_folds]
    mean_acc = sum(accs) / len(accs)
    std_acc = (sum((a - mean_acc) ** 2 for a in accs) / len(accs)) ** 0.5

    summary = {
        "dataset": args.dataset, "epochs": n_epochs, "extra_args": args.extra,
        "best_epoch_0indexed": best_epoch, "mean_test_acc_at_best_epoch": mean_acc,
        "std_test_acc_at_best_epoch": std_acc, "per_fold_acc_at_best_epoch": accs,
        "total_runtime_sec": time.time() - t0,
    }
    out_path.write_text(json.dumps(
        {"dataset": args.dataset, "epochs": args.epochs, "extra_args": args.extra,
         "folds": all_folds, "summary": summary}, indent=2))
    print(json.dumps(summary, indent=2))


if __name__ == "__main__":
    main()


## 4. Run: IMDB-BINARY (paper's original data)

In [ ]:
!python run_gin.py --dataset IMDBBINARY --epochs 350 --extra --degree_as_tag

## 5. Run: IMDB-MULTI (paper's original data)

In [ ]:
!python run_gin.py --dataset IMDBMULTI --epochs 350 --extra --degree_as_tag

## 6. Run: IMDBBINARY-NEW (our scraped data)

In [ ]:
!python run_gin.py --dataset IMDBBINARY-NEW --epochs 350 --extra --degree_as_tag

## 7. Run: IMDBMULTI-NEW (our scraped data)

In [ ]:
!python run_gin.py --dataset IMDBMULTI-NEW --epochs 350 --extra --degree_as_tag

## 8. Final summary — copy this whole block back into the chat

In [ ]:
import json
import glob

print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
for f in sorted(glob.glob("results/*_cv_curves.json")):
    d = json.load(open(f))
    s = d.get("summary")
    if s:
        print(f"{s['dataset']:20s} mean={s['mean_test_acc_at_best_epoch']*100:5.1f}%  "
              f"std={s['std_test_acc_at_best_epoch']*100:4.1f}%  "
              f"best_epoch={s['best_epoch_0indexed']+1:3d}  "
              f"runtime={s['total_runtime_sec']/60:5.1f} min")
    else:
        print(f"{f}: no summary yet (run incomplete)")


## 9. (Optional) Download raw results to bring back

In [ ]:
from google.colab import files
import shutil

shutil.make_archive("gin_results", "zip", "results")
files.download("gin_results.zip")
